# This Notebook will read the Supples Messagses and extract the sentiments using AI Models

In [1]:
external_data_catalog="demo_finance_external"
silver_schema="demo_finance_external_schema"
external_data_catalog_folder_path="/Volumes/demo_finance_external/demo_finance_external_schema/demo_finance_external_vol"
gold_catalog      = "demo_fusion_gold"
gold_schema       = "aidp"

# Read the supplier communication file and run AI Model


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, concat, col

# ----------------------------
# Step 0: Initialize Spark
# ----------------------------
spark = SparkSession.builder.appName("SupplierSentimentLLM").getOrCreate()

# ----------------------------
# Step 1: Load supplier sentiment CSV
# ----------------------------
data_path = "/Volumes/demo_finance_external/demo_finance_external_schema/demo_finance_external_vol/Fusion Supplier Communications.txt"  # <- update path

df = spark.read.option("header", True).csv(f"{external_data_catalog_folder_path}/Fusion Supplier Communications.txt")

# ----------------------------
# Step 2: Call LLM to extract sentiment and aspect
# ----------------------------
df_llm = df.withColumn(
    "sentiment_analysis",
    expr("""
        query_model(
            'cohere.command-plus-latest',
            concat(
                'Analyze the following supplier comment. Return JSON with two keys: ',
                'sentiment (Positive, Negative, Neutral) and aspect (delivery, quality, invoice, packaging, communication, general). ',
                'Return ONLY a compact one-line JSON, no extra text. Comment: ', invoice_comment
            ),
            map('max_output_tokens', 200)
        )
    """)
)

# ----------------------------
# Step 3: Show LLM output
# ----------------------------
df_llm.select("supplier_id", "invoice_comment", "sentiment_analysis").show(truncate=False)


+---------------+------------------------------------------------------------------------------+--------------------------------------------------------+
|supplier_id    |invoice_comment                                                               |sentiment_analysis                                      |
+---------------+------------------------------------------------------------------------------+--------------------------------------------------------+
|300000047414571|Minor delay in invoice approval, but overall good service and timely delivery.|{"sentiment":"Positive","aspect":"invoice"}             |
|300000047414571|Product quality exceeded expectations, very satisfied with the supplier.      |{"sentiment": "Positive", "aspect": "quality"}          |
|300000047414571|Late shipment, but supplier responded quickly to resolve the issue.           |{"sentiment": "Positive", "aspect": "delivery"}         |
|300000047414571|Invoice approved on time, smooth transaction overall.      

# Format the Data to insert into Gold catalog

In [3]:
from pyspark.sql.functions import col, split, explode, trim
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import from_json

# ----------------------------
# Step 1: Define schema for the LLM output JSON
# ----------------------------
json_schema = StructType([
    StructField("sentiment", StringType(), True),
    StructField("aspect", StringType(), True)
])

# ----------------------------
# Step 2: Parse the JSON column into separate columns
# ----------------------------
df_parsed = df_llm.withColumn(
    "parsed",
    from_json("sentiment_analysis", json_schema)
).select(
    "supplier_id",
    "invoice_comment",
    col("parsed.sentiment").alias("sentiment"),
    col("parsed.aspect").alias("aspect")
)

# ----------------------------
# Step 3: Split comma-separated aspects into array
# ----------------------------
df_split = df_parsed.withColumn(
    "aspect_array",
    split(col("aspect"), ",\s*")  # split on comma + optional space
)

# ----------------------------
# Step 4: Explode array into separate rows
# ----------------------------
df_exploded = df_split.withColumn(
    "aspect_individual",
    explode(col("aspect_array"))
)

# ----------------------------
# Step 5: Trim whitespace and drop intermediate columns
# ----------------------------
df_exploded = df_exploded.withColumn("aspect_individual", trim(col("aspect_individual")))
df_exploded = df_exploded.drop("aspect", "aspect_array")

# ----------------------------
# Step 6: Show final normalized DataFrame
# ----------------------------
df_exploded.show(truncate=False)


+---------------+------------------------------------------------------------------------------+---------+-----------------+
|supplier_id    |invoice_comment                                                               |sentiment|aspect_individual|
+---------------+------------------------------------------------------------------------------+---------+-----------------+
|300000047414571|Minor delay in invoice approval, but overall good service and timely delivery.|Positive |invoice          |
|300000047414571|Minor delay in invoice approval, but overall good service and timely delivery.|Positive |delivery         |
|300000047414571|Product quality exceeded expectations, very satisfied with the supplier.      |Positive |quality          |
|300000047414571|Late shipment, but supplier responded quickly to resolve the issue.           |Positive |delivery         |
|300000047414571|Invoice approved on time, smooth transaction overall.                         |Positive |invoice          |


In [4]:
df_exploded.write.insertInto(f"{gold_catalog}.{gold_schema}.SUPPLIER_SENTIMENT_ANALYSIS")